<h1>This notebook is to calculate the F1 score and mAP score for the combination of YOLO localisation and ResNet classification</h1>

In [36]:
import os
import json
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torch import optim
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from PIL import Image
import time



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

import json

def dump_json(json_data, json_path):
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(json_data, f, indent=4, ensure_ascii=False)

def read_json(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        return json.load(f)

Device: cpu


We first load our YOLO model and classification model

In [37]:
from ultralytics import YOLO

yolo_model = YOLO('all_images_best_with_occlusion.pt')

In [38]:
class CropModel(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        for param in backbone.parameters():
            param.requires_grad = False
        for param in backbone.layer4.parameters():
            param.requires_grad = True # Freezes all layers except layer4

        # extracts final feature layer before class score in ResNet
        self.feature_layers = nn.Sequential(*list(backbone.children())[:-1])
        
        # Custom Layouts According to previous papers
        self.fc1 = nn.Linear(2048, 1024)
        self.bn1 = nn.BatchNorm1d(1024)
        self.relu = nn.ReLU()
        self.dropout1 = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(1024, 512)
        self.bn2 = nn.BatchNorm1d(512)
        self.dropout2 = nn.Dropout(p=0.3)

        # Category classifier
        self.category_classifier = nn.Linear(512, 13)
    
    def forward(self, x):

        #ResNet50
        tensor = self.feature_layers(x)      # Shape: [batch, 2048, 1, 1]
        x = torch.flatten(tensor, 1)         # Shape: [batch, 2048]
        
        # Custom Layout according to previous papers
        x = self.fc1(x)                      # [batch, 1024]
        x = self.bn1(x)                      # [batch, 1024]
        x = self.relu(x)                     # [batch, 1024]
        x = self.dropout1(x)                 # [batch, 1024]
        
        x = self.fc2(x)                      # [batch, 512]
        x = self.bn2(x)                      # [batch, 512]
        x = self.relu(x)                     # [batch, 512]
        x = self.dropout2(x)                 # [batch, 512]
        
        category_out = self.category_classifier(x)  # [batch, 13]
        return category_out

In [39]:
model = CropModel()
model.load_state_dict(torch.load('classification_model.pth', map_location='cpu'))
model.to(device)

CropModel(
  (feature_layers): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
          (0): Co

<h1>We first calculate the F1 score for classification ONLY</h1>(we will combine both localisation and classification later)

F1 = (2*TP)/(2TP+FP+FN)

TP = true positive
FP = false positive
FN = false negative

In [41]:
from PIL import Image
import torch
import torchvision.transforms as transforms

def predict_image(model, image_path, device):
    """
    Predict the category of a single image.
    
    Args:
        model: Loaded CropModel
        image_path: Path to the image file
        device: 'cuda' or 'cpu'
    
    Returns:
        predicted_category: Integer (0-12, representing category 1-13)
    """
    # Define the same transform used during training (UPDATE THIS IF THE TRANSFORMATION CHANGES)
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0)
    image_tensor = image_tensor.to(device)
    
    # Predict
    model.eval()  # Set to evaluation mode
    with torch.no_grad():
        outputs = model(image_tensor)
        _, predicted = torch.max(outputs, 1)
    
    return predicted.item()  # Returns 0-12

In [52]:
#initalise f1 scores dictionary
f1_scores = {
    "true_positive_count":0, "false_positive_count":0, "false_negative_count":0, "f1":0,
    "occlusion1_true_positive_count":0, "occlusion1_false_positive_count":0, "occlusion1_false_negative_count":0, "f1_occlusion1":0,
    "occlusion2_true_positive_count":0, "occlusion2_false_positive_count":0, "occlusion2_false_negative_count":0, "f1_occlusion2":0,
    "occlusion3_true_positive_count":0, "occlusion3_false_positive_count":0, "occlusion3_false_negative_count":0, "f1_occlusion3":0
}

for category_id in range(13):
    f1_scores[category_id] = {
        "true_positive_count":0, "false_positive_count":0, "false_negative_count":0, "f1":0,
        "occlusion1_true_positive_count":0, "occlusion1_false_positive_count":0, "occlusion1_false_negative_count":0, "f1_occlusion1":0,
        "occlusion2_true_positive_count":0, "occlusion2_false_positive_count":0, "occlusion2_false_negative_count":0, "f1_occlusion2":0,
        "occlusion3_true_positive_count":0, "occlusion3_false_positive_count":0, "occlusion3_false_negative_count":0, "f1_occlusion3":0
    }

def update_f1_scores(f1_scores):
    temp = f1_scores
    temp['f1'] = 2*temp['true_positive_count'] / (2*temp['true_positive_count'] + temp['false_positive_count'] + temp['false_negative_count']) if 2*temp['true_positive_count'] + temp['false_positive_count'] + temp['false_negative_count'] > 0 else 'null'
    temp['f1_occlusion1'] = 2*temp['occlusion1_true_positive_count'] / (2*temp['occlusion1_true_positive_count'] + temp['occlusion1_false_positive_count'] + temp['occlusion1_false_negative_count']) if 2*temp['occlusion1_true_positive_count'] + temp['occlusion1_false_positive_count'] + temp['occlusion1_false_negative_count'] > 0 else 'null'
    temp['f1_occlusion2'] = 2*temp['occlusion2_true_positive_count'] / (2*temp['occlusion2_true_positive_count'] + temp['occlusion2_false_positive_count'] + temp['occlusion2_false_negative_count']) if 2*temp['occlusion2_true_positive_count'] + temp['occlusion2_false_positive_count'] + temp['occlusion2_false_negative_count'] > 0 else 'null'
    temp['f1_occlusion3'] = 2*temp['occlusion3_true_positive_count'] / (2*temp['occlusion3_true_positive_count'] + temp['occlusion3_false_positive_count'] + temp['occlusion3_false_negative_count']) if 2*temp['occlusion3_true_positive_count'] + temp['occlusion3_false_positive_count'] + temp['occlusion3_false_negative_count'] > 0 else 'null'
    for category_id in range(13):
        temp = f1_scores[category_id]
        temp['f1'] = 2*temp['true_positive_count'] / (2*temp['true_positive_count'] + temp['false_positive_count'] + temp['false_negative_count']) if 2*temp['true_positive_count'] + temp['false_positive_count'] + temp['false_negative_count'] > 0 else 'null'
        temp['f1_occlusion1'] = 2*temp['occlusion1_true_positive_count'] / (2*temp['occlusion1_true_positive_count'] + temp['occlusion1_false_positive_count'] + temp['occlusion1_false_negative_count']) if 2*temp['occlusion1_true_positive_count'] + temp['occlusion1_false_positive_count'] + temp['occlusion1_false_negative_count'] > 0 else 'null'
        temp['f1_occlusion2'] = 2*temp['occlusion2_true_positive_count'] / (2*temp['occlusion2_true_positive_count'] + temp['occlusion2_false_positive_count'] + temp['occlusion2_false_negative_count']) if 2*temp['occlusion2_true_positive_count'] + temp['occlusion2_false_positive_count'] + temp['occlusion2_false_negative_count'] > 0 else 'null'
        temp['f1_occlusion3'] = 2*temp['occlusion3_true_positive_count'] / (2*temp['occlusion3_true_positive_count'] + temp['occlusion3_false_positive_count'] + temp['occlusion3_false_negative_count']) if 2*temp['occlusion3_true_positive_count'] + temp['occlusion3_false_positive_count'] + temp['occlusion3_false_negative_count'] > 0 else 'null'
            

In [54]:

#get all images into variable
import os
image_dir = "stage2_crops_val"
all_imgs = sorted([f for f in os.listdir(image_dir)])
image_files = [f for f in all_imgs if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print(f"Total images: {len(image_files)}")

#iterate each image
size = len(image_files)
i = 0
for image in image_files:
    i +=1
    if i % 100 == 0:
        print(f"{str(i)}/{str(size)}",end='\r')
    #get ground truths
    filetag = image.split('_')[0]
    itemtag = image.split('_')[1].split('.')[0]
    file_json = read_json(f'val/annos/{filetag}.json')
    item_json = file_json[itemtag]
    true_category_id = item_json['category_id']-1
    occlusion = item_json['occlusion']

    #get prediction
    file_directory = f"{image_dir}/{filetag}_{itemtag}.jpg"
    predicted_category_id = predict_image(model, file_directory, device)

    # update f1 scores
    # check if true positive
    if predicted_category_id == true_category_id:
        f1_scores['true_positive_count'] += 1
        f1_scores[true_category_id]['true_positive_count'] += 1
        if occlusion > 0:
            f1_scores[f'occlusion{str(occlusion)}_true_positive_count'] += 1
            f1_scores[true_category_id][f'occlusion{str(occlusion)}_true_positive_count'] += 1
            
    
    if predicted_category_id != true_category_id:
        f1_scores[predicted_category_id]['false_positive_count'] += 1 #update false positives per class
        f1_scores['false_positive_count'] += 1 #update false positives overall
        if occlusion > 0:
            f1_scores[predicted_category_id][f'occlusion{str(occlusion)}_false_positive_count'] += 1 #update false positives per class
            f1_scores[f'occlusion{str(occlusion)}_false_positive_count'] += 1 #update false positives overall
            
        
        f1_scores[true_category_id]['false_negative_count'] += 1 #update false negatives per class
        f1_scores['false_negative_count'] += 1 #update false negatives overall
        if occlusion > 0:
            f1_scores[true_category_id][f'occlusion{str(occlusion)}_false_negative_count'] += 1 #update false negatives per class
            f1_scores[f'occlusion{str(occlusion)}_false_negative_count'] += 1 #update false negatives overall
            

    update_f1_scores(f1_scores)

Total images: 136
100/136

In [64]:
import pandas as pd

# Extract overall F1 scores
overall_data = {
    'Category': 'OVERALL',
    'All Occlusions': f1_scores['f1'],
    'Occlusion 1': f1_scores['f1_occlusion1'],
    'Occlusion 2': f1_scores['f1_occlusion2'],
    'Occlusion 3': f1_scores['f1_occlusion3']
}

# Extract per-category F1 scores
category_data = []
for cat_id in range(13):
    cat_f1 = f1_scores[cat_id]['f1']
    occ1_f1 = f1_scores[cat_id]['f1_occlusion1']
    occ2_f1 = f1_scores[cat_id]['f1_occlusion2']
    occ3_f1 = f1_scores[cat_id]['f1_occlusion3']
    
    # Convert 'null' strings to None
    cat_f1 = None if cat_f1 == 'null' else cat_f1
    occ1_f1 = None if occ1_f1 == 'null' else occ1_f1
    occ2_f1 = None if occ2_f1 == 'null' else occ2_f1
    occ3_f1 = None if occ3_f1 == 'null' else occ3_f1
    
    category_data.append({
        'Category': f'Class {cat_id}',
        'All Occlusions': cat_f1,
        'Occlusion 1': occ1_f1,
        'Occlusion 2': occ2_f1,
        'Occlusion 3': occ3_f1
    })

# Combine and create DataFrame
all_data = [overall_data] + category_data
df = pd.DataFrame(all_data)

# Format F1 scores as percentages (using loc to avoid chained assignment warning)
for col in ['All Occlusions', 'Occlusion 1', 'Occlusion 2', 'Occlusion 3']:
    df[col] = df[col].apply(lambda x: f"{x*100:.1f}%" if isinstance(x, (int, float)) and not pd.isna(x) else 'N/A')

# Display the table
print("\nF1 SCORES BY CATEGORY AND OCCLUSION LEVEL\n")
print(df.to_string(index=False))


F1 SCORES BY CATEGORY AND OCCLUSION LEVEL

Category All Occlusions Occlusion 1 Occlusion 2 Occlusion 3
 OVERALL          28.7%       28.2%       27.9%       50.0%
 Class 0          44.0%       40.0%       42.9%      100.0%
 Class 1          28.6%       28.6%       28.6%         N/A
 Class 2            N/A         N/A         N/A         N/A
 Class 3            N/A         N/A         N/A         N/A
 Class 4           0.0%        0.0%        0.0%         N/A
 Class 5            N/A         N/A         N/A         N/A
 Class 6          28.6%       50.0%        0.0%        0.0%
 Class 7          25.0%       66.7%       18.2%        0.0%
 Class 8          24.5%       18.2%       30.8%        0.0%
 Class 9           0.0%        0.0%        0.0%         N/A
Class 10          25.0%        0.0%       66.7%         N/A
Class 11           0.0%        0.0%        0.0%         N/A
Class 12           0.0%         N/A        0.0%         N/A


/var/folders/gp/v3fj2md118b8nm60k2q1y5k80000gn/T/ipykernel_3919/547746856.py:40: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df[col] = df[col].apply(lambda x: f"{x*100:.1f}%" if isinstance(x, (int, float)) and not pd.isna(x) else 'N/A')


<h1>Secondly, we calculate the mAP score for localisation AND classification</h1>